# 06. Pembelajaran transfer untuk teks

Melatih model sumber, memindahkan embedding, membekukan bobot, lalu melakukan fine-tuning pada domain sasaran. Seluruh eksperimen inti dapat dijalankan tanpa unduhan model eksternal.

**Prasyarat:** modul 05.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Sumber dan sasaran

Pembelajaran transfer menggunakan parameter hasil pembelajaran sebelumnya. Demonstrasi ini memindahkan embedding dari ulasan produk ke ulasan layanan. Keduanya berasal dari data sintetis. Ini memperlihatkan mekanismenya, bukan ukuran manfaat model bahasa pralatih berskala besar.

In [2]:
source_rows = read_rows(domain="produk")
target_rows = read_rows(domain="layanan")
# Kosakata sumber dibentuk dari data latih sumber saja.
vocab,source_train,source_val,source_test = loaders(source_rows)
_,target_train,target_val,target_test = loaders(target_rows,vocab=vocab)
print("Sumber:",len(source_rows),"Sasaran:",len(target_rows))
print("Token domain sasaran:",encode("pelayanan",vocab))

Sumber: 240 Sasaran: 240
Token domain sasaran: [1]


## 2. Melatih model sumber

Checkpoint sumber dipilih dengan validasi sumber. Vocabulary sumber tetap digunakan pada sasaran. Kata domain baru menjadi UNK. Model subword dapat mengurangi masalah ini, tetapi tidak menghapus seluruh pergeseran domain.

In [3]:
seed_all(42)
source = MeanClassifier(len(vocab),dim=16)
fit(source,source_train,source_val,epochs=15)
print("Validasi sumber:",run_epoch(source,source_val))
pretrained = source.embedding.weight.detach().clone()

Validasi sumber: {'loss': 0.447026789188385, 'accuracy': 0.75, 'macro_f1': 0.7333333492279053, 'confusion': [[12, 12], [0, 24]]}


## 3. Memindahkan dan membekukan embedding

Salin bobot embedding, buat kepala klasifikasi baru, lalu set requires_grad=False pada embedding. Optimizer hanya menerima parameter yang boleh berubah. Ini disebut ekstraksi fitur dengan representasi tetap.

In [4]:
seed_all(42)
frozen = MeanClassifier(len(vocab),dim=16)
with torch.no_grad():
    frozen.embedding.weight.copy_(pretrained)
frozen.embedding.weight.requires_grad_(False)
fit(frozen,target_train,target_val,epochs=15)
assert torch.equal(frozen.embedding.weight,pretrained)
print("Validasi frozen:",run_epoch(frozen,target_val))

Validasi frozen: {'loss': 0.4074491659800212, 'accuracy': 0.75, 'macro_f1': 0.75, 'confusion': [[18, 6], [6, 18]]}


## 4. Membuka bobot untuk fine-tuning

Fine-tuning memperbarui parameter yang sebelumnya dipelajari. Kita menyalin model frozen agar kedua kondisi dapat dibandingkan. Laju belajar lebih kecil membatasi perubahan awal, tetapi tetap harus dinilai melalui validasi.

In [5]:
import copy
finetuned = copy.deepcopy(frozen)
finetuned.embedding.weight.requires_grad_(True)
fit(finetuned,target_train,target_val,epochs=10,lr=0.001)
print("Perubahan embedding:",(finetuned.embedding.weight-pretrained).norm().item())
print("Validasi fine-tuning:",run_epoch(finetuned,target_val))

Perubahan embedding: 0.5704332590103149
Validasi fine-tuning: {'loss': 0.39811190962791443, 'accuracy': 0.75, 'macro_f1': 0.75, 'confusion': [[18, 6], [6, 18]]}


## 5. Pembanding dari awal

Manfaat transfer perlu dibandingkan dengan model yang dilatih dari awal menggunakan domain sasaran. Samakan data sasaran, arsitektur, dan seed. Total komputasi sumber tetap perlu dilaporkan. Jangan menjanjikan transfer selalu lebih baik.

In [6]:
seed_all(42)
_,fresh_train,fresh_val,_ = loaders(target_rows,vocab=vocab,seed=42)
scratch = MeanClassifier(len(vocab),dim=16)
fit(scratch,fresh_train,fresh_val,epochs=15)
candidates = {"frozen":frozen,"finetuned":finetuned,"scratch":scratch}
validation = {name:run_epoch(m,target_val) for name,m in candidates.items()}
print(validation)
best = min(validation,key=lambda name:validation[name]["loss"])
print("Terpilih:",best,"Uji sasaran:",run_epoch(candidates[best],target_test))

{'frozen': {'loss': 0.4074491659800212, 'accuracy': 0.75, 'macro_f1': 0.75, 'confusion': [[18, 6], [6, 18]]}, 'finetuned': {'loss': 0.39811190962791443, 'accuracy': 0.75, 'macro_f1': 0.75, 'confusion': [[18, 6], [6, 18]]}, 'scratch': {'loss': 0.41075243552525836, 'accuracy': 0.75, 'macro_f1': 0.75, 'confusion': [[18, 6], [6, 18]]}}
Terpilih: finetuned Uji sasaran: {'loss': 0.40793153643608093, 'accuracy': 0.75, 'macro_f1': 0.7460317611694336, 'confusion': [[15, 9], [3, 21]]}


## 6. Hubungan dengan BERT

Pada praktik model bahasa pralatih, bobot sumber diperoleh dari korpus jauh lebih besar dengan tujuan seperti masked language modeling. Tokenizer harus berasal dari checkpoint yang sama. Model klasifikasi kemudian memakai kepala tugas yang dapat dilatih.

Notebook 10 menyediakan kode eksplisit untuk memuat tokenizer dan BERT kecil melalui Transformers, melakukan fine-tuning dengan DataLoader PyTorch, mengevaluasi data tahanan lokal, dan menyimpan model. Jalur itu memerlukan internet serta dependensi tambahan.

## Latihan mandiri

1. Apa perbedaan freezing dan eval()?
2. Mengapa tidak membentuk kosakata sasaran baru lalu langsung memakai bobot sumber?
3. Apakah sumber boleh memakai data uji sasaran?
4. Mengapa perlu pembanding dari awal?

## Pembahasan latihan

1. Freezing menonaktifkan gradien parameter. eval() mengubah perilaku lapisan seperti dropout.
2. Urutan indeks dapat berubah. Bobot harus dipetakan ulang berdasarkan token jika vocabulary diperluas.
3. Tidak untuk protokol evaluasi ini, karena menyebabkan kebocoran informasi.
4. Transfer dapat memberi manfaat kecil atau bahkan menurunkan kinerja pada sasaran.

## Penghubung ke materi berikutnya

Modul 07 mencatat perbandingan model dan seed agar kesimpulan tidak didasarkan satu hasil yang kebetulan baik.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.